# Practical No. 10 — Design and Deploy a Deep Learning Model Using Flask or Streamlit

## Objectives
- Understand deployment of deep learning models.
- Train and save a neural network model.
- Create a web application using Flask or Streamlit.
- Deploy the model for user interaction and predictions.

## Theory
**Model deployment** makes a trained model available through web applications, APIs or cloud platforms. The document’s workflow is: Train Model → Save Model → Create Web Application → Load Saved Model → Accept User Input → Generate Prediction → Display Results.

**Flask** is a lightweight Python web framework for web applications and REST APIs. **Streamlit** is a framework for quickly building interactive data science and ML applications.

## Software Requirements
- Python 3.x
- TensorFlow / Keras
- Flask or Streamlit
- NumPy
- Pandas
- Scikit-learn

### Problem Statement
Deploy a handwritten digit recognition model trained on MNIST.

## Algorithm
1. Load MNIST dataset.
2. Preprocess data.
3. Build a neural network.
4. Train and save the model.
5. Create Flask/Streamlit application and load saved model.
6. Accept user input, make prediction, and display it.

## Part A: Training the Deep Learning Model
### Step 1: Import libraries

In [1]:
import tensorflow as tf
import numpy as np
from pathlib import Path
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
print("TensorFlow:", tf.__version__)

TensorFlow: 2.22.0-rc0


### Step 2: Load dataset and normalize images

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
print("Training:", X_train.shape, "Test:", X_test.shape)

Training: (60000, 28, 28) Test: (10000, 28, 28)


### Step 3: Build model
Architecture follows the Word document: Flatten → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax).

In [3]:
model = Sequential([
    tf.keras.Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

### Step 4: Compile model

In [4]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

### Step 5: Train model
The source specifies **5 epochs**. Results depend on the runtime and training.

In [5]:
history = model.fit(X_train, y_train, epochs=5, batch_size=128, verbose=1)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9098 - loss: 0.3225
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9606 - loss: 0.1329
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9728 - loss: 0.0919
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9799 - loss: 0.0700
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9842 - loss: 0.0551


### Step 6: Evaluate model

In [6]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", loss)
print("Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9728 - loss: 0.0889
Test Loss: 0.08893705159425735
Accuracy: 0.9728000164031982


### Step 7: Save the trained model in modern and portable formats

The source document saves `mnist_model.h5`. Because you encountered a Keras version mismatch while loading that format, this updated implementation saves `mnist_model.keras` and numeric weights (`mnist_weights.npz`). The network architecture and experiment remain the same.

In [7]:
from pathlib import Path
import numpy as np

MODEL_PATH = Path('mnist_model.keras')
WEIGHTS_PATH = Path('mnist_weights.npz')
model.save(MODEL_PATH)
weights = model.get_weights()
np.savez_compressed(WEIGHTS_PATH, **{f'arr_{i}': w for i, w in enumerate(weights)})
print('Saved:', MODEL_PATH.resolve())
print('Saved:', WEIGHTS_PATH.resolve())

Saved: D:\DL-Precticals\mnist_model.keras
Saved: D:\DL-Precticals\mnist_weights.npz


### Step 8: Check saved weights by reconstructing the same network

This test does not require deserializing the full Keras model. It will also catch missing or mismatched weights before deploying.

In [8]:
restored = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
with np.load(WEIGHTS_PATH, allow_pickle=False) as saved:
    keys = sorted(saved.files, key=lambda k: int(k.split('_')[1]))
    restored.set_weights([saved[k] for k in keys])
p1 = model.predict(X_test[:2], verbose=0)
p2 = restored.predict(X_test[:2], verbose=0)
np.testing.assert_allclose(p1, p2, rtol=1e-5, atol=1e-6)
print('Model weight export validated.')

Model weight export validated.


### Download model files (Google Colab only)

If training in Colab, download both files and copy `mnist_weights.npz` to the deployment project root.

In [9]:
try:
    from google.colab import files
except ImportError:
    print('Local environment: saved files are in the current notebook directory.')
else:
    files.download(str(MODEL_PATH))
    files.download(str(WEIGHTS_PATH))

Local environment: saved files are in the current notebook directory.


## Part B: Deployment Using Flask

The provided deployment ZIP includes the Flask application, HTML template, `model_utils.py`, and `requirements.txt`. The application loads the weights from the package root rather than using a hard-coded `D:` path.

**Run from PowerShell in the extracted deployment folder:**
```powershell
python -m pip install -r requirements.txt
cd flask_app
python app.py
```
Open `http://127.0.0.1:5000`. Export `mnist_weights.npz` from Part A and put it in the package root **before** running.

## Part C: Deployment Using Streamlit

The Streamlit application performs the same pixel-value demonstration as in the Word document, with the model files resolved relative to the deployment folder.

**From the extracted deployment folder:**
```powershell
cd streamlit_app
python -m streamlit run app.py
```
Open the local address shown in the terminal (usually `http://localhost:8501`).

## Deployment on Cloud
The Word document names GitHub, Streamlit Cloud, Heroku, AWS EC2, Google Cloud Platform and Microsoft Azure as possible platforms.

## Result
The MNIST model is trained, evaluated, and exported for deployment by Flask and Streamlit. The practical is complete after the two applications successfully start and return predictions. Training accuracy and predictions depend on actual execution, not the sample outputs in the source document.

**Compatibility fix:** Both applications reconstruct the document’s model architecture and load `mnist_weights.npz`, avoiding incompatible whole-model deserialization across Keras versions.

**Important:** The constant-intensity input in the document is a deployment demonstration rather than a real handwritten-digit input.